<a href="https://colab.research.google.com/github/shoh0806/Deep-Learning-Project/blob/main/ClickBait.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#0. 라이브러리 import

In [3]:
"""
YouTube 썸네일 + 제목 수집 스크립트
- mtv.csv  → label = 1 (클릭베이트)
- nmtv.csv → label = 0 (정상)
- 썸네일: API 없이 URL로 직접 다운로드
- 제목: YouTube Data API v3 사용
"""

import pandas as pd
import requests
import os
import time
from google.colab import userdata

#00. 설정

In [4]:
# 방법 1. GitHub에서 직접 다운로드 (추천 ✅)
# 세션 끊겨도 다시 실행하면 자동으로 받아와요
# !wget -q https://raw.githubusercontent.com/wajihanaveed/ThumbnailTruth/main/mtv.csv
# !wget -q https://raw.githubusercontent.com/wajihanaveed/ThumbnailTruth/main/nmtv.csv

# 방법 2. 직접 업로드 (아래 주석 해제해서 사용)
from google.colab import files
uploaded = files.upload()  # mtv.csv, nmtv.csv 둘 다 선택
print("CSV 다운로드 완료!")


Saving mtv.csv to mtv (1).csv
Saving nmtv.csv to nmtv (1).csv
CSV 다운로드 완료!


#000 설정

In [5]:
MTV_CSV = "mtv.csv"
NMTV_CSV = "nmtv.csv"
THUMBNAIL_DIR = "thumbnails"
OUTPUT_CSV = "dataset.csv"
API_KEY = userdata.get('YOUTUBE_API_KEY')  # Colab 비밀키에서 불러오기

os.makedirs(THUMBNAIL_DIR, exist_ok=True)

#1. URL에서 Video ID 추출

In [6]:
def extract_video_id(url):
    """
    https://www.youtube.com/watch?v=lepHJI4ze-w
    → lepHJI4ze-w
    """
    try:
        return url.strip().split("v=")[1]
    except:
        return None

# 2. 두 CSV 합치고 라벨 붙이기

In [7]:
mtv_df = pd.read_csv(MTV_CSV)
nmtv_df = pd.read_csv(NMTV_CSV)

mtv_df["label"] = 1   # 클릭베이트
nmtv_df["label"] = 0  # 정상

df = pd.concat([mtv_df, nmtv_df], ignore_index=True)
df["video_id"] = df["url"].apply(extract_video_id)

# video_id 추출 실패한 행 제거
df = df.dropna(subset=["video_id"])
df = df.reset_index(drop=True)

print(f"총 데이터 수: {len(df)}")
print(f"클릭베이트(1): {len(df[df['label']==1])}개")
print(f"정상(0): {len(df[df['label']==0])}개")
print(df.head())

총 데이터 수: 2843
클릭베이트(1): 1359개
정상(0): 1484개
                                           url  label     video_id
0  https://www.youtube.com/watch?v=lepHJI4ze-w      1  lepHJI4ze-w
1  https://www.youtube.com/watch?v=ezwXAlJ5pAY      1  ezwXAlJ5pAY
2  https://www.youtube.com/watch?v=i6_OomloWUg      1  i6_OomloWUg
3  https://www.youtube.com/watch?v=VvuuR7HifUY      1  VvuuR7HifUY
4  https://www.youtube.com/watch?v=uwIjrEbywnk      1  uwIjrEbywnk


# 3. 썸네일 다운로드 (API 불필요)

In [8]:
def download_thumbnail(video_id, save_dir):
    """
    유튜브 썸네일을 URL로 직접 다운로드
    화질 우선순위: maxresdefault > hqdefault > mqdefault
    """
    qualities = ["maxresdefault", "hqdefault", "mqdefault"]

    for quality in qualities:
        url = f"https://img.youtube.com/vi/{video_id}/{quality}.jpg"
        try:
            response = requests.get(url, timeout=10)
            if response.status_code == 200:
                save_path = os.path.join(save_dir, f"{video_id}.jpg")
                with open(save_path, "wb") as f:
                    f.write(response.content)
                return save_path
        except:
            continue

    return None  # 모든 화질 실패 시



# 4. 제목 수집 (YouTube Data API v3)

In [9]:
def get_video_title(video_id, api_key):
    """
    YouTube Data API v3로 제목 가져오기
    삭제된 영상이면 None 반환
    """
    url = "https://www.googleapis.com/youtube/v3/videos"
    params = {
        "part": "snippet",
        "id": video_id,
        "key": api_key
    }

    try:
        response = requests.get(url, params=params, timeout=10)
        data = response.json()

        # 삭제된 영상 or 비공개 영상
        if not data.get("items"):
            return None

        return data["items"][0]["snippet"]["title"]
    except:
        return None

# 5. 전체 수집 실행

In [10]:
results = []
skipped = 0

for idx, row in df.iterrows():
    video_id = row["video_id"]
    label = row["label"]

    print(f"[{idx+1}/{len(df)}] {video_id} 처리 중...", end=" ")

    # 썸네일 다운로드
    thumbnail_path = download_thumbnail(video_id, THUMBNAIL_DIR)

    # 제목 수집
    title = get_video_title(video_id, API_KEY)

    # 삭제된 영상 스킵
    if thumbnail_path is None or title is None:
        print("→ 스킵 (삭제된 영상)")
        skipped += 1
        continue

    print(f"→ 완료 | {title[:30]}...")

    results.append({
        "video_id": video_id,
        "title": title,
        "thumbnail_path": thumbnail_path,
        "label": label
    })

    time.sleep(0.1)  # API 과부하 방지

[1/2843] lepHJI4ze-w 처리 중... → 완료 | I Bought 250 BANNED Amazon Pro...
[2/2843] ezwXAlJ5pAY 처리 중... → 완료 | TikTok Wigofellas PRANKS on MO...
[3/2843] i6_OomloWUg 처리 중... → 완료 | TikTok Wigofellas PRANKS on MO...
[4/2843] VvuuR7HifUY 처리 중... → 완료 | Superheroes in Real Life Caugh...
[5/2843] uwIjrEbywnk 처리 중... → 완료 | 20 People You Won't Believe Ex...
[6/2843] RLkdkdQI-ZU 처리 중... → 완료 | Best 22 SCIENCE EXPERIMENTS Co...
[7/2843] 7g9xcCMdwns 처리 중... → 완료 | Spider-Man Rated-R...
[8/2843] rSXyLqvlUA0 처리 중... → 완료 | If You Build It, I'll Buy It!...
[9/2843] _18BtKjb4MI 처리 중... → 스킵 (삭제된 영상)
[10/2843] yjex6KjvJ4M 처리 중... → 완료 | Funniest Moments in Women's Fo...
[11/2843] rgBVLnBSgHs 처리 중... → 완료 | Full stop punctuation...
[12/2843] XGPex66EyTU 처리 중... → 완료 | BREAKING 100 RULES IN 24 HOURS...
[13/2843] Ls5yMv96vXo 처리 중... → 완료 | I Was MURDERED…...
[14/2843] 6E7nxP3iD4c 처리 중... → 완료 | I Became the World's Worst Bab...
[15/2843] IGS04NYSG0c 처리 중... → 완료 | हॉट बनाम कोल्ड फूड चैलेंज | पा...
[16/2843

# 6. 최종 데이터셋 저장

In [11]:
result_df = pd.DataFrame(results)
result_df.to_csv(OUTPUT_CSV, index=False)

print(f"\n========== 수집 완료 ==========")
print(f"최종 수집: {len(result_df)}개")
print(f"스킵(삭제된 영상): {skipped}개")
print(f"저장 위치: {OUTPUT_CSV}")
print(result_df.head())


========== 수집 완료 ==========
최종 수집: 2512개
스킵(삭제된 영상): 331개
저장 위치: dataset.csv
      video_id                                              title  \
0  lepHJI4ze-w               I Bought 250 BANNED Amazon Products!   
1  ezwXAlJ5pAY  TikTok Wigofellas PRANKS on MOM - Wigofellas P...   
2  i6_OomloWUg  TikTok Wigofellas PRANKS on MOM - Wigofellas P...   
3  VvuuR7HifUY        Superheroes in Real Life Caught On Camera !   
4  uwIjrEbywnk  20 People You Won't Believe Existed Till You S...   

               thumbnail_path  label  
0  thumbnails/lepHJI4ze-w.jpg      1  
1  thumbnails/ezwXAlJ5pAY.jpg      1  
2  thumbnails/i6_OomloWUg.jpg      1  
3  thumbnails/VvuuR7HifUY.jpg      1  
4  thumbnails/uwIjrEbywnk.jpg      1  
